<a href="https://colab.research.google.com/github/omami1155/ENDFIELD-kishitu-tool/blob/main/%E8%B6%85%E5%9F%9F%E6%B4%BB%E6%80%A7%E7%82%B9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from dataclasses import dataclass
from itertools import combinations
from typing import Dict, FrozenSet, Iterable, Literal, Tuple
from collections import defaultdict

import ipywidgets as widgets
from IPython.display import display, clear_output

固定スロット = Literal["付加効果", "スキル効果"]

基礎効果ID_to_名前: Dict[int, str] = {
    1: "敏捷UP",
    2: "筋力UP",
    3: "意志UP",
    4: "知性UP",
    5: "メイン能力UP",
}

付加効果ID_to_名前: Dict[int, str] = {
    1: "攻撃力UP",
    2: "物理ダメージUP",
    3: "灼熱ダメージUP",
    4: "電磁ダメージUP",
    5: "寒冷ダメージUP",
    6: "自然ダメージUP",
    7: "会心率UP",
    8: "必殺技効率UP",
    9: "アーツ強度UP",
    10: "アーツダメージUP",
    11: "回復効率UP",
    12: "HPアップ",
}

スキル効果ID_to_名前: Dict[int, str] = {
    1: "強攻",
    2: "圧制",
    3: "追襲",
    4: "破砕",
    5: "巧技",
    6: "噴発",
    7: "流回",
    8: "効率",
    9: "昂揚",
    10: "付術",
    11: "治癒",
    12: "切骨",
    13: "残虐",
    14: "夜幕",
}

基礎効果名_to_ID: Dict[str, int] = {v: k for k, v in 基礎効果ID_to_名前.items()}
付加効果名_to_ID: Dict[str, int] = {v: k for k, v in 付加効果ID_to_名前.items()}
スキル効果名_to_ID: Dict[str, int] = {v: k for k, v in スキル効果ID_to_名前.items()}

def 表示_基礎(i: int) -> str:
    return 基礎効果ID_to_名前.get(i, f"未定義({i})")

def 表示_付加(i: int) -> str:
    return 付加効果ID_to_名前.get(i, f"未定義({i})")

def 表示_スキル(i: int) -> str:
    return スキル効果ID_to_名前.get(i, f"未定義({i})")

@dataclass(frozen=True)
class 武器:
    name: str
    基礎効果: int
    付加効果: int
    スキル効果: int

@dataclass(frozen=True)
class ダンジョン:
    name: str
    出る基礎効果: FrozenSet[int]
    出る付加効果: FrozenSet[int]
    出るスキル効果: FrozenSet[int]

@dataclass(frozen=True)
class 絞り込み:
    dungeon: str
    基礎効果候補: FrozenSet[int]
    固定する枠: 固定スロット
    固定する効果: int

@dataclass(frozen=True)
class 周回プラン:
    dungeon: str
    絞り: 絞り込み
    同時に狙える武器: Tuple[武器, ...]
    スコア: Tuple[int, int]

DUNGEONS: dict[str, ダンジョン] = {
    "中枢エリア": ダンジョン(
        name="中枢エリア",
        出る基礎効果=frozenset({1, 2, 3, 4, 5}),
        出る付加効果=frozenset({1, 3, 4, 5, 6, 8, 9, 10}),
        出るスキル効果=frozenset({1, 2, 3, 4, 5, 6, 7, 8}),
    ),
    "原石研究パーク": ダンジョン(
        name="原石研究パーク",
        出る基礎効果=frozenset({1, 2, 3, 4, 5}),
        出る付加効果=frozenset({1, 2, 4, 5, 6, 7, 8, 10}),
        出るスキル効果=frozenset({2, 3, 5, 8, 9, 10, 11, 12}),
    ),
    "鉱山エリア": ダンジョン(
        name="鉱山エリア",
        出る基礎効果=frozenset({1, 2, 3, 4, 5}),
        出る付加効果=frozenset({2, 3, 5, 6, 7, 9, 11, 12}),
        出るスキル効果=frozenset({1, 2, 5, 6, 8, 10, 13, 14}),
    ),
    "エネルギー高地": ダンジョン(
        name="エネルギー高地",
        出る基礎効果=frozenset({1, 2, 3, 4, 5}),
        出る付加効果=frozenset({1, 2, 3, 6, 7, 9, 11, 12}),
        出るスキル効果=frozenset({3, 4, 7, 9, 10, 11, 12, 13}),
    ),
    "武陵城": ダンジョン(
        name="武陵城",
        出る基礎効果=frozenset({1, 2, 3, 4, 5}),
        出る付加効果=frozenset({1, 4, 5, 7, 8, 10, 11, 12}),
        出るスキル効果=frozenset({1, 4, 6, 7, 11, 12, 13, 14}),
    ),
    "清波砦": ダンジョン(
        name="清波砦",
        出る基礎効果=frozenset({1, 2, 3, 4, 5}),
        出る付加効果=frozenset({2, 4, 5, 8, 9, 10, 11, 12}),
        出るスキル効果=frozenset({2, 4, 5, 6, 9, 11, 12, 14}),
    )
}

WEAPONS: list[武器] = [
    武器("片手剣-鋼鉄余音", 1, 2, 5),
    武器("片手剣-堅城鋳造者", 4, 8, 9),
    武器("片手剣-フィンチェイサー3.0", 2, 5, 2),
    武器("片手剣-十二問", 1, 1, 10),
    武器("片手剣-O.B.J.軽刃", 1, 1, 7),
    武器("片手剣-仰止", 1, 2, 14),
    武器("片手剣-大願", 1, 1, 10),
    武器("片手剣-不知帰", 3, 1, 7),
    武器("片手剣-フレイムフォージ", 4, 1, 14),
    武器("片手剣-ダークトーチ", 4, 3, 10),
    武器("片手剣-フーヤオ", 5, 7, 14),
    武器("片手剣-テルミット·カッター", 3, 1, 7),
    武器("片手剣-輝かしき名声", 5, 2, 13),
    武器("片手剣-白夜新星", 5, 9, 10),
    武器("片手剣-栄光の記憶", 1, 7, 14),

    武器("大剣-探龍", 2, 8, 6),
    武器("大剣-千古恒常", 2, 9, 13),
    武器("大剣-最期の声", 2, 12, 11),
    武器("大剣-O.B.J.重責", 2, 12, 8),
    武器("大剣-大雷斑", 2, 12, 11),
    武器("大剣-クラヴェンガー", 2, 1, 6),
    武器("大剣-鑑", 5, 1, 2),
    武器("大剣-昔日の逸品", 3, 12, 8),
    武器("大剣-破砕君主", 2, 7, 4),

    武器("長柄武器-正義嵌合", 2, 8, 13),
    武器("長柄武器-O.B.J.鋭矛", 3, 2, 10),
    武器("長柄武器-求心の槍", 3, 4, 2),
    武器("長柄武器-負山", 1, 2, 8),
    武器("長柄武器-勇猛", 1, 2, 5),
    武器("長柄武器-J.E.T.", 5, 1, 2),

    武器("拳銃-作品:衆生", 1, 10, 10),
    武器("拳銃-O.B.J.迅速", 1, 8, 6),
    武器("拳銃-合理的決別", 2, 3, 3),
    武器("拳銃-芸術の独裁者", 4, 7, 12),
    武器("拳銃-ナビゲーター", 4, 5, 10),
    武器("拳銃-楔", 5, 7, 10),
    武器("拳銃-同類共食", 5, 10, 10),
    武器("拳銃-落草", 1, 1, 6),
    武器("拳銃-望郷", 1, 5, 2),

    武器("アーツユニット-弔いの詩", 4, 1, 14),
    武器("アーツユニット-術無", 3, 8, 9),
    武器("アーツユニット-荒野迷走", 4, 4, 10),
    武器("アーツユニット-布教の自由", 3, 11, 11),
    武器("アーツユニット-O.B.J.術識", 4, 9, 3),
    武器("アーツユニット-使命必達", 3, 8, 3),
    武器("アーツユニット-蒼星の囁き", 4, 11, 10),
    武器("アーツユニット-作品:蝕跡", 3, 6, 2),
    武器("アーツユニット-破壊ユニット", 5, 9, 6),
    武器("アーツユニット-遺忘", 4, 10, 14),
    武器("アーツユニット-騎士精神", 3, 12, 11),
]

def 正解基質がダンジョンで出る(w: 武器, d: ダンジョン) -> bool:
    return (
        (w.基礎効果 in d.出る基礎効果) and
        (w.付加効果 in d.出る付加効果) and
        (w.スキル効果 in d.出るスキル効果)
    )

def 絞り込みで正解が拾える(w: 武器, d: ダンジョン, f: 絞り込み) -> bool:
    if f.dungeon != d.name:
        return False
    if not 正解基質がダンジョンで出る(w, d):
        return False
    if w.基礎効果 not in f.基礎効果候補:
        return False
    if f.固定する枠 == "付加効果":
        return w.付加効果 == f.固定する効果
    return w.スキル効果 == f.固定する効果

def 武器名から周回プランを提案(
    weapon_name: str,
    weapons: Iterable[武器],
    dungeons: dict[str, ダンジョン],
    top_n: int = 5,
) -> list[周回プラン]:
    target = next((w for w in weapons if w.name == weapon_name), None)
    if target is None:
        return []

    plans: list[周回プラン] = []

    for d in dungeons.values():
        if not 正解基質がダンジョンで出る(target, d):
            continue

        pool = [w for w in weapons if 正解基質がダンジョンで出る(w, d)]

        slots: Tuple[固定スロット, ...] = ("付加効果", "スキル効果")
        for fixed_slot in slots:
            fixed_value = target.付加効果 if fixed_slot == "付加効果" else target.スキル効果

            if fixed_slot == "付加効果" and fixed_value not in d.出る付加効果:
                continue
            if fixed_slot == "スキル効果" and fixed_value not in d.出るスキル効果:
                continue

            same_fixed = [
                w for w in pool
                if (w.付加効果 == fixed_value if fixed_slot == "付加効果" else w.スキル効果 == fixed_value)
            ]
            possible_bases = sorted({w.基礎効果 for w in same_fixed})

            for k in (1, 2, 3):
                for combo in combinations(possible_bases, k):
                    if target.基礎効果 not in combo:
                        continue

                    f = 絞り込み(
                        dungeon=d.name,
                        基礎効果候補=frozenset(combo),
                        固定する枠=fixed_slot,
                        固定する効果=fixed_value,
                    )

                    matched = tuple(w for w in pool if 絞り込みで正解が拾える(w, d, f))
                    if target not in matched:
                        continue

                    score = (len(matched), -len(f.基礎効果候補))
                    plans.append(周回プラン(
                        dungeon=d.name,
                        絞り=f,
                        同時に狙える武器=matched,
                        スコア=score,
                    ))

    plans.sort(key=lambda x: (-x.スコア[0], x.dungeon, x.スコア[1], x.絞り.固定する枠, x.絞り.固定する効果))
    return plans[:top_n]

def 周回プランを文字で表示(plans: list[周回プラン], target_name: str) -> None:
    if not plans:
        print("プランが見つかりませんでした。")
        return

    print(f"検索武器: {target_name}")
    print("==================================================")

    for pl in plans:
        base_list = " / ".join(表示_基礎(i) for i in sorted(pl.絞り.基礎効果候補))
        fixed_value_str = 表示_付加(pl.絞り.固定する効果) if pl.絞り.固定する枠 == "付加効果" else 表示_スキル(pl.絞り.固定する効果)
        others = [w.name for w in pl.同時に狙える武器 if w.name != target_name]

        print(f"\n■ {pl.dungeon}")
        print(f"  基礎効果候補: {base_list}")
        print(f"  {pl.絞り.固定する枠}固定: {fixed_value_str}")
        if others:
            print(f"  一緒に狙える: {', '.join(others)}")
        else:
            print("  他に同時に狙える武器はなし")

def 基質から武器を逆引き(基礎: int, 付加: int, スキル: int, weapons: Iterable[武器]) -> list[武器]:
    out: list[武器] = []
    for w in weapons:
        if w.基礎効果 == 基礎 and w.付加効果 == 付加 and w.スキル効果 == スキル:
            out.append(w)
    out.sort(key=lambda x: x.name)
    return out

def 武器一覧を武器種で表示(ws: list[武器]) -> None:
    if not ws:
        print("一致する武器はありません。")
        return
    grp: dict[str, list[str]] = defaultdict(list)
    for w in ws:
        t, n = w.name.split("-", 1)
        grp[t].append(n)
    for t in sorted(grp.keys()):
        print(f"\n■ {t}")
        for n in sorted(grp[t]):
            print(f"  - {t}-{n}")

武器種ごと = defaultdict(list)
for w in WEAPONS:
    武器種, _武器名 = w.name.split("-", 1)
    武器種ごと[武器種].append(w.name)

武器種一覧 = sorted(武器種ごと.keys())

武器種ドロップ = widgets.Dropdown(options=武器種一覧, description="武器種:")
武器名ドロップ = widgets.Dropdown(description="武器名:")
表示件数スライダー = widgets.IntSlider(value=5, min=1, max=30, step=1, description="表示件数:")
検索ボタン = widgets.Button(description="検索")
出力エリア1 = widgets.Output()

def 武器種変更時(change):
    選択種 = change["new"]
    opts = sorted(武器種ごと[選択種])
    武器名ドロップ.options = opts
    if opts:
        武器名ドロップ.value = opts[0]

武器種ドロップ.observe(武器種変更時, names="value")

初期opts = sorted(武器種ごと[武器種一覧[0]])
武器名ドロップ.options = 初期opts
武器名ドロップ.value = 初期opts[0]

def 検索実行(_):
    with 出力エリア1:
        clear_output()
        weapon_name = 武器名ドロップ.value
        n = 表示件数スライダー.value
        plans = 武器名から周回プランを提案(weapon_name, WEAPONS, DUNGEONS, top_n=n)
        周回プランを文字で表示(plans, weapon_name)

検索ボタン.on_click(検索実行)

基礎ドロップ = widgets.Dropdown(options=sorted(基礎効果名_to_ID.keys()), description="基礎:")
付加ドロップ = widgets.Dropdown(options=sorted(付加効果名_to_ID.keys()), description="付加:")
スキルドロップ = widgets.Dropdown(options=sorted(スキル効果名_to_ID.keys()), description="スキル:")
逆引きボタン = widgets.Button(description="逆引き検索")
出力エリア2 = widgets.Output()

def 逆引き実行(_):
    with 出力エリア2:
        clear_output()
        b = 基礎効果名_to_ID[基礎ドロップ.value]
        a = 付加効果名_to_ID[付加ドロップ.value]
        s = スキル効果名_to_ID[スキルドロップ.value]
        ws = 基質から武器を逆引き(b, a, s, WEAPONS)
        print(f"選択基質: {基礎ドロップ.value} / {付加ドロップ.value} / {スキルドロップ.value}")
        print("==================================================")
        武器一覧を武器種で表示(ws)

逆引きボタン.on_click(逆引き実行)

tab = widgets.Tab()
tab.children = [
    widgets.VBox([武器種ドロップ, 武器名ドロップ, 表示件数スライダー, 検索ボタン, 出力エリア1]),
    widgets.VBox([基礎ドロップ, 付加ドロップ, スキルドロップ, 逆引きボタン, 出力エリア2]),
]
tab.set_title(0, "武器→周回プラン")
tab.set_title(1, "基質→武器逆引き")

display(tab)
